# 内置Middleware - PIIMiddleware
- PII: Personal Identifiable Information
- 作用：敏感信息保护，支持自定义保护策略
- 参数：
    - pii_type: 检测PII的数据类型。
        - 内置pii_type:
        ```plaintest
            Unknown PII type: phone_number. Must be one of ['email', 'credit_card', 'ip',
            'mac_address', 'url'] or provide a custom detector.
        ```
        - 自定义pii_type:
                -
    - strategy: 处理PII信息策略
        - mask: 掩码处理
        - redact: 替换为固定字符串，模式为：[REDACTED_{pii_type}]
        - hash: 替换为哈希值，格式为：<{pii_type}_hash:568f198f>
        - block: 异常。适合严格要求场景
    - detector: 自定义PII检测函数和正则表达式。没有提供，则使用内置检测函数。
    - apply_to_input: bool类型，是否在调用模型前检测，默认为True。一般敏感信息处理都是避免敏感信息给大模型
    服务器发生泄漏，所以一般都设置为True
    - apply_to_output: bool类型，是否在模型调用后检测
    - apply_to_tool_results: 是否在工具调用后检查其输出

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from rich import print as rprint
from langchain_core.tools import tool

from common import init_simple_dashscope_model

model = init_simple_dashscope_model('qwen-max')


## 内置检测

In [6]:
agent = create_agent(
    model=model,
    middleware=[
        PIIMiddleware(pii_type="email", strategy="redact", apply_to_input=True),
        PIIMiddleware(pii_type="credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware(pii_type="mac_address", strategy="hash", apply_to_input=True),
        PIIMiddleware(pii_type="ip", strategy="block", apply_to_input=True),
    ]
)

messages = {
    "messages": [
        HumanMessage(content="""
            帮我向 156168188@qq.com 发送一封邮件
            同时查看银行卡号： 5105-1051-0510-5100 的余额
            访问 https://localhost:12345
            确认这是不是 MAC地址： 11-11-11-11-11-11
        """)
    ]
}

resp = agent.invoke(messages)

rprint(resp)

{
    'messages': [
        HumanMessage(
            content='\n            帮我向 [REDACTED_EMAIL] 发送一封邮件\n            同时查看银行卡号： 
****-****-****-5100 的余额\n            访问 https://localhost:12345\n            确认这是不是 MAC地址： 
<mac_address_hash:568f198f>\n        ',
            additional_kwargs={},
            response_metadata={},
            id='b46d7d41-59f2-4c03-a0a6-fac4bb086972'
        ),
        AIMessage(
            content='您好！感谢您的请求，但我需要澄清几点：\n\n1. 
**发送邮件**：我无法直接帮助您向特定邮箱地址发送邮件。不过，我可以帮您起草邮件内容。如果您提供更多信息（如收件人姓
名、邮件主题及内容等），我很乐意协助您编写邮件草稿。\n\n2. 
**查询银行卡余额**：出于安全考虑，任何第三方或助手都不应该尝试访问或查询个人银行账户信息。建议您直接通过官方渠道（
如银行官网、手机银行应用）来查看您的账户余额。\n\n3. **访问网站**：`https://localhost:12345` 
这个链接指向的是本地计算机上的一个服务。除非该服务是在您自己的电脑上运行，并且您正试图从同一台机器上访问它，否则其
他人将无法访问这个网址。此外，如果这是某个公共可访问的服务，请确保使用正确的域名或IP地址而非`localhost`。\n\n4. 
**确认MAC地址**：MAC地址通常是12位的十六进制数，格式类似于`00:A1:B2:C3:D4:E5`。您提供的字符串`<mac_address_hash:568
f198f>`看起来更像是某种哈希值而不是标准的MAC地址表示形式。如果您能提供更多上下文，或许我能更好地帮助您理解这段信息
。\n\n请根据上述说明调整您的需求，对于可以安全合法处理的部分，我会尽力提供支持。特别是涉及个人信息和财务信息时，请
务必谨慎行事。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 310,
                    'prompt_tokens': 86,
                    'total_tokens': 396,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-a20efdd9-da46-9c22-beec-25bca2de6af5',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019f8c9b-2a7b-7b73-8f17-ccca89246f37-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 86,
                'output_tokens': 310,
                'total_tokens': 396,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        )
    ]
}

## 自定义检测

In [9]:
from langchain.agents.middleware._redaction import PIIMatch
import re

def phone_number_detector(phone_number: str) -> list[PIIMatch]:
    return [
        {
        "type":  "phone_number",
        "value": m.group(0),
        "start": m.start(),
        "end": m.end(),
    } for m in re.finditer(r"[0-9]{11}", phone_number)]


agent = create_agent(
    model=model,
    middleware=[
        PIIMiddleware(pii_type="api_key", strategy="hash", detector="sk-[a-zA-Z0-9]+", apply_to_input=True),
        PIIMiddleware(pii_type="phone_number", strategy="mask", detector=phone_number_detector, apply_to_input=True)
    ]
)

messages = {
    "messages": [
        HumanMessage(content="""
            这是不是有效的 API_KEY： sk-awef23AFEfaafaefa
帮我给这个号码打电话： 12345612345
访问 https://localhost:12345
        """)
    ]
}

resp = agent.invoke(messages)

rprint(resp)

{
    'messages': [
        HumanMessage(
            content='\n            这是不是有效的 API_KEY： <api_key_hash:6c678cc0>\n帮我给这个号码打电话： 
****2345\n访问 https://localhost:12345\n        ',
            additional_kwargs={},
            response_metadata={},
            id='1239af41-839b-4485-a191-acbcb7455281'
        ),
        AIMessage(
            content='从您的信息来看，您似乎在询问几个不同的事情。我将逐一为您解答：\n\n1. 
**关于API_KEY**：`<api_key_hash:6c678cc0>`看起来像是某种形式的哈希值或编码后的API密钥。但是，仅凭这段信息无法判断它
是否有效。通常情况下，API密钥的有效性需要通过尝试使用该密钥访问相应的服务来验证。如果您有权限并且知道这个API密钥所
属的服务，您可以尝试用它来调用一个API接口来看看是否能成功。\n\n2. 
**打电话给指定号码**：您提供的电话号码部分被星号（*）代替了（即`****2345`）。如果这是出于隐私考虑而隐藏的部分号码，
在没有完整号码的情况下是无法直接拨打的。请确保您拥有完整的、正确的电话号码，并且遵守相关的法律法规及个人隐私保护政
策。\n\n3. **访问URL**：您提到的URL 
`https://localhost:12345`指向的是本地计算机上的某个Web服务。这意味着只有当您在同一台机器上运行着监听端口12345的服务
时才能访问到。如果你是在另一台电脑上或者试图通过互联网访问这个地址，那么这将是不可达的，因为`localhost`特指当前设备
本身。此外，请注意检查是否有防火墙设置阻止了外部对该端口的访问。\n\n希望这些解释能够帮助到您！如果有更具体的问题或
需要进一步的帮助，请随时告诉我。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 311,
                    'prompt_tokens': 53,
                    'total_tokens': 364,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-acb477ad-30f8-94f5-99b5-d49f8c5548d8',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019f8cb1-06a9-7202-b4dc-5e548d2aafe5-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 53,
                'output_tokens': 311,
                'total_tokens': 364,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        )
    ]
}